# M0 Baseline Benchmark: Beat/Miss Naive Rule

The simplest possible earnings prediction strategy:

- **Rule**: If EPS beat consensus (`surprise_pct > 0`), predict stock goes **Up**; if miss, predict **Down**.
- **No model training** — just a direct mapping from the earnings headline.

This notebook evaluates M0 standalone to establish the floor that any learned model must clear.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.size'] = 12

conn = sqlite3.connect('../data/hackathon.db')
print('DB connected')

## 1. Build Target Variable

For each of the 91 earnings events, compute the 5-trading-day cumulative return after the earnings date. Label = 1 if positive (stock went up), 0 otherwise.

In [ ]:
earnings = pd.read_sql('SELECT * FROM earnings', conn)
prices = pd.read_sql('SELECT * FROM daily_prices', conn)
prices['date'] = pd.to_datetime(prices['date'])
earnings['earnings_date'] = pd.to_datetime(earnings['earnings_date'])

records = []
for _, evt in earnings.iterrows():
    tk = evt['ticker']
    ed = evt['earnings_date']
    
    tk_prices = prices[prices['ticker'] == tk].sort_values('date')
    post = tk_prices[tk_prices['date'] > ed].head(5)
    pre = tk_prices[tk_prices['date'] <= ed].tail(1)
    if len(post) < 5 or pre.empty:
        continue
    
    p0 = pre['close'].values[0]
    p5 = post['close'].values[-1]
    ret_5d = (p5 - p0) / p0
    
    records.append({
        'ticker': tk,
        'earnings_date': ed,
        'surprise_pct': evt['surprise_pct'],
        'ret_5d': ret_5d,
        'target': 1 if ret_5d > 0 else 0,
    })

df = pd.DataFrame(records).sort_values('earnings_date').reset_index(drop=True)
print(f'Total events: {len(df)}')
print(f'Target distribution: {df["target"].value_counts().to_dict()}')
print(f'Up ratio: {df["target"].mean():.1%}')
df.head(10)

---

## 2. M0 — Naive Beat/Miss Rule

Apply M0 on the same test window as all other models (events 41–91, walk-forward `MIN_TRAIN=40`).

In [ ]:
MIN_TRAIN = 40

y = df['target'].values
m0_pred = (df['surprise_pct'].values > 0).astype(int)

# Test set: events from MIN_TRAIN onward
m0_pred_test = m0_pred[MIN_TRAIN:]
y_test = y[MIN_TRAIN:]
test_df = df.iloc[MIN_TRAIN:].copy()

m0_acc = accuracy_score(y_test, m0_pred_test)
m0_f1 = f1_score(y_test, m0_pred_test)
m0_auc = roc_auc_score(y_test, m0_pred_test.astype(float))
cm = confusion_matrix(y_test, m0_pred_test)

# Precision / Recall
tp, fp, tn, fn = cm[1, 1], cm[0, 1], cm[0, 0], cm[1, 0]
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0

print('=' * 55)
print('M0 Naive Rule: Beat → Up, Miss → Down')
print('=' * 55)
print(f'Test events:  {len(y_test)}')
print(f'Accuracy:     {m0_acc:.3f}')
print(f'Precision:    {precision:.3f}')
print(f'Recall:       {recall:.3f}')
print(f'F1 Score:     {m0_f1:.3f}')
print(f'AUC-ROC:      {m0_auc:.3f}')
print()
print('Classification Report:')
print(classification_report(y_test, m0_pred_test, target_names=['Down', 'Up'], zero_division=0))
print('Confusion Matrix (rows=actual, cols=predicted):')
print(f'           Pred Down  Pred Up')
print(f'  Actual Down    {tn:3d}      {fp:3d}')
print(f'  Actual Up      {fn:3d}      {tp:3d}')
print()

beat_count = (m0_pred_test == 1).sum()
miss_count = (m0_pred_test == 0).sum()
print(f'Predictions: {beat_count} beats (→Up), {miss_count} misses (→Down)')
print(f'Beat ratio in test set: {beat_count}/{len(y_test)} = {beat_count/len(y_test):.1%}')
print(f'When beat: {tp}/{beat_count} actually went up ({tp/beat_count:.1%})')
if miss_count > 0:
    print(f'When miss: {tn}/{miss_count} actually went down ({tn/miss_count:.1%})')

---

## 3. Per-Ticker Breakdown

How does M0 perform for each M7 company?

In [ ]:
test_df = test_df.copy()
test_df['m0_pred'] = m0_pred_test
test_df['m0_correct'] = (m0_pred_test == y_test)

ticker_stats = []
for tk in sorted(test_df['ticker'].unique()):
    t = test_df[test_df['ticker'] == tk]
    n = len(t)
    correct = t['m0_correct'].sum()
    beats = (t['m0_pred'] == 1).sum()
    actual_up = (t['target'] == 1).sum()
    ticker_stats.append({
        'Ticker': tk,
        'Events': n,
        'Beats': beats,
        'Actual Up': actual_up,
        'M0 Correct': correct,
        'Accuracy': f'{correct/n:.0%}',
    })

ticker_df = pd.DataFrame(ticker_stats)
print(ticker_df.to_string(index=False))
print()
# Which tickers does M0 fail on most?
worst = ticker_df.sort_values('M0 Correct').head(3)
print('Hardest tickers for M0 (beat ≠ up):')
for _, r in worst.iterrows():
    print(f"  {r['Ticker']}: {r['M0 Correct']}/{r['Events']} correct ({r['Accuracy']})")

---

## 4. M0 Baseline Benchmark Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('M0 Baseline Benchmark: Beat/Miss Naive Rule', fontsize=15, fontweight='bold')

# ── 1. Confusion Matrix Heatmap ──
ax = axes[0, 0]
cm_display = np.array([[tn, fp], [fn, tp]])
im = ax.imshow(cm_display, cmap='Blues', aspect='auto')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Pred Down', 'Pred Up'])
ax.set_yticklabels(['Actual Down', 'Actual Up'])
ax.set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        val = cm_display[i, j]
        total_row = cm_display[i].sum()
        ax.text(j, i, f'{val}\n({val/total_row:.0%})', ha='center', va='center',
                fontsize=14, fontweight='bold',
                color='white' if val > cm_display.max() * 0.6 else 'black')

# ── 2. Metrics Bar Chart ──
ax = axes[0, 1]
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'AUC']
values = [m0_acc, precision, recall, m0_f1, m0_auc]
colors = ['#F44336', '#FF9800', '#4CAF50', '#FF9800', '#2196F3']
bars = ax.bar(metrics, values, color=colors, alpha=0.8, width=0.6)
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f'{v:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random')
ax.set_ylim(0, 1.1)
ax.set_title('Model Metrics vs Random Baseline')
ax.legend()
ax.grid(axis='y', alpha=0.3)

# ── 3. EPS Surprise vs Post-Earnings Return ──
ax = axes[1, 0]
correct_mask = test_df['m0_correct']
ax.scatter(test_df.loc[correct_mask, 'surprise_pct'],
           test_df.loc[correct_mask, 'ret_5d'] * 100,
           c='#4CAF50', alpha=0.6, s=50, label=f'Correct ({correct_mask.sum()})', zorder=3)
ax.scatter(test_df.loc[~correct_mask, 'surprise_pct'],
           test_df.loc[~correct_mask, 'ret_5d'] * 100,
           c='#F44336', marker='x', alpha=0.7, s=50, label=f'Wrong ({(~correct_mask).sum()})', zorder=3)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_xlabel('EPS Surprise %')
ax.set_ylabel('5-Day Return %')
ax.set_title('EPS Surprise vs Post-Earnings Return')
ax.legend(loc='upper left')
# Quadrant labels
ax.text(0.95, 0.95, 'Beat & Up\n(M0 ✓)', transform=ax.transAxes, ha='right', va='top', fontsize=9, color='green', alpha=0.6)
ax.text(0.05, 0.95, 'Miss & Up\n(M0 ✗)', transform=ax.transAxes, ha='left', va='top', fontsize=9, color='red', alpha=0.6)
ax.text(0.95, 0.05, 'Beat & Down\n(M0 ✗)', transform=ax.transAxes, ha='right', va='bottom', fontsize=9, color='red', alpha=0.6)
ax.text(0.05, 0.05, 'Miss & Down\n(M0 ✓)', transform=ax.transAxes, ha='left', va='bottom', fontsize=9, color='green', alpha=0.6)
ax.grid(alpha=0.3)

# ── 4. Cumulative Trading Returns ──
ax = axes[1, 1]
test_df['m0_strat_ret'] = np.where(m0_pred_test == 1, test_df['ret_5d'], -test_df['ret_5d'])
test_df['cum_m0'] = (1 + test_df['m0_strat_ret']).cumprod()
test_df['cum_buyhold'] = (1 + test_df['ret_5d']).cumprod()

m0_gross = (test_df['cum_m0'].iloc[-1] - 1) * 100
bh_gross = (test_df['cum_buyhold'].iloc[-1] - 1) * 100

x_range = range(len(test_df))
ax.plot(x_range, test_df['cum_m0'].values, '#F44336', linewidth=2,
        label=f'M0 Strategy ({m0_gross:+.1f}%)')
ax.plot(x_range, test_df['cum_buyhold'].values, 'gray', linewidth=1.5, alpha=0.6,
        label=f'Buy & Hold ({bh_gross:+.1f}%)')
ax.axhline(y=1.0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Event #')
ax.set_ylabel('Cumulative Return')
ax.set_title('Simulated Trading Returns')
ax.legend(loc='upper left')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../outputs/m0_baseline_benchmark.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/m0_baseline_benchmark.png')

---

## 5. Summary

| Metric | M0 (Beat/Miss Rule) | Random |
|--------|---------------------|--------|
| Accuracy | 0.549 | 0.500 |
| Precision | 0.522 | — |
| Recall | 0.960 | — |
| F1 | 0.676 | — |
| AUC-ROC | 0.557 | 0.500 |
| Gross Return | +66.6% | — |

**Key takeaways:**

1. **M0 almost always predicts "Up"** — M7 companies beat earnings in ~90% of test events, so the rule is structurally biased toward "Up."
2. **High recall (0.960), low precision (0.522)** — it catches nearly every actual rise, but half of its "Up" predictions are wrong (beat + drop events).
3. **+66.6% gross return** — despite low precision, the strategy profits because it's long-biased in a generally rising market. The few correct short calls add marginal edge.
4. **The "Beat But Drop" problem** — 22 out of 46 beat events resulted in the stock dropping. This is the gap that sentiment features (M2) will attempt to fill.